# Practice Lab: Model Selection & Routing (Lesson 13.3)

Module 13 . 4 exercises . Benchmark + domain rules + cache combo + adaptive routing


# Lesson 13.3: Model Selection & Routing

4 exercises: 1E + 2M + 1C

Module 13: Cost Optimization & Efficiency


In [ ]:
import re, random
from collections import defaultdict
random.seed(42)


---
## Exercise 1 (Easy): Benchmark Tasks


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
MODELS={"flash":{"in":0.10,"out":0.40},"flash25":{"in":0.15,"out":0.60},"pro":{"in":1.25,"out":10.00}}
TASKS={"greeting":{"f":9.5,"p":9.5,"t":15},"factual":{"f":9.0,"p":9.2,"t":20},"summarize":{"f":8.0,"p":9.0,"t":15},
    "code_simple":{"f":8.5,"p":9.3,"t":10},"analysis":{"f":6.5,"p":9.0,"t":12},"code_complex":{"f":5.5,"p":8.8,"t":8},
    "reasoning":{"f":5.0,"p":9.0,"t":10},"creative":{"f":6.0,"p":8.5,"t":10}}

print("Quality-Cost Frontier:")
print(f"  {'Cat':<14} {'Flash':>6} {'Pro':>6} {'Gap':>5} {'Verdict':<20} {'Traf%':>6}")
ft=f25t=pt=0
for c,i in TASKS.items():
    g=i["p"]-i["f"]
    if g<=1: v="Flash OK"; ft+=i["t"]
    elif g<=2: v="Flash25 sweet spot"; f25t+=i["t"]
    else: v="Pro needed"; pt+=i["t"]
    print(f"  {c:<14} {i['f']:>5.1f} {i['p']:>5.1f} {g:>4.1f}  {v:<20} {i['t']:>5}%")

def mc(mk,pct):
    m=MODELS[mk]; r=10000*pct/100
    return r*(500*m["in"]/1e6+300*m["out"]/1e6)*30

rc=mc("flash",ft)+mc("flash25",f25t)+mc("pro",pt)
ap=mc("pro",100); sv=ap-rc; sp=sv/ap*100
print(f"\n  Traffic: Flash {ft}% | Flash25 {f25t}% | Pro {pt}%")
print(f"  Routing: ${rc:.2f}/mo | Always-Pro: ${ap:.2f}/mo | Savings: ${sv:.2f} ({sp:.0f}%)")


</details>


---
## Exercise 2 (Medium): Domain Rules


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

class DRC:
    SIMPLE=[(r"^(hi|hello|hey|good\s*(morning|afternoon|evening))[!.\s]*$","greeting"),
        (r"^(thanks|thank you|ok|bye)[!.\s]*$","greeting"),
        (r"^(what|who|when|where)\s+(is|are|was|were)\s+\w+","factual"),
        (r"^(define|meaning of)","factual"),(r"^(yes|no|true|false)[.\s]*$","greeting"),
        (r"(how much|price|cost|fee)\s+(is|of|for)","pricing"),
        (r"(how (do|can) i|steps to)\s+(enroll|register|sign up)","enrollment"),
        (r"(what|which)\s+course","course_lookup"),
        (r"(refund|cancel|payment)\s*(policy|process)?","faq")]
    COMPLEX=[(r"(compare|contrast|versus|vs\b|difference between).+and\s+","analysis"),
        (r"(analyze|evaluate|assess|critique)\s+(this|the|my)","analysis"),
        (r"(design|architect|build|implement)\s+(a |an )?(system|arch|platform)","code_complex"),
        (r"(why|how)\s+(does|do|would|should).{30,}","reasoning"),
        (r"(write|generate)\s+(a |an )?(report|essay|article).{20,}","creative"),
        (r"(step.by.step|think through|explain.+in detail)","reasoning")]
    def classify(self,q):
        q=q.strip().lower(); w=len(q.split())
        if w<=5:
            for p,c in self.SIMPLE:
                if re.search(p,q,re.I): return {"tier":"flash","cat":c,"m":"rule"}
            return {"tier":"flash","cat":"short","m":"wc"}
        for p,c in self.SIMPLE:
            if re.search(p,q,re.I): return {"tier":"flash","cat":c,"m":"rule"}
        for p,c in self.COMPLEX:
            if re.search(p,q,re.I): return {"tier":"pro","cat":c,"m":"rule"}
        if w>80: return {"tier":"pro","cat":"long","m":"wc"}
        return None

r=DRC()
qs=[("Hello!","flash"),("Thanks!","flash"),("What is Python?","flash"),("How much does GenAI cost?","flash"),
    ("How do I enroll?","flash"),("What course for AI?","flash"),("Refund policy?","flash"),
    ("Define ML","flash"),("Yes","flash"),("Bye!","flash"),
    ("Compare RAG vs fine-tuning and recommend the best approach for my use case","pro"),
    ("Design a microservice architecture for a chat platform with 10K users","pro"),
    ("Analyze this code for performance issues and suggest fixes","pro"),
    ("Why does gradient descent converge to local minima and how can you avoid it in deep learning?","pro"),
    ("Write a comprehensive report on AI regulation in India covering DPDP Act","pro"),
    ("Step by step explain how transformers work","pro"),
    ("Summarize this article","ambiguous"),("Write Flask hello world","ambiguous"),
    ("Explain recursion","ambiguous"),("Help me debug this","ambiguous")]

print("Domain Rules:")
print(f"  Rules: {len(r.SIMPLE)} simple + {len(r.COMPLEX)} complex")
correct=classified=0
for q,exp in qs:
    res=r.classify(q)
    if res: classified+=1; got=res["tier"]
    else: got="ambig"
    if got==exp: correct+=1
print(f"  Classified: {classified}/{len(qs)} ({classified/len(qs)*100:.0f}% free)")
print(f"  Accuracy: {correct}/{len(qs)} ({correct/len(qs)*100:.0f}%)")


</details>


---
## Exercise 3 (Medium): Router+Cache Combo


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class RCC:
    def __init__(self,cr=0.35): self.cr=cr; self.s={"t":0,"ch":0,"fl":0,"f25":0,"pro":0}
    def process(self,qt):
        self.s["t"]+=1
        if random.random()<self.cr: self.s["ch"]+=1; return 0
        routing={"greeting":"fl","factual":"fl","pricing":"fl","enrollment":"fl",
            "summarize":"f25","code_simple":"f25","analysis":"pro","code_complex":"pro",
            "reasoning":"pro","creative":"pro"}
        tier=routing.get(qt,"f25"); self.s[tier]+=1
        costs={"fl":0.000170,"f25":0.000255,"pro":0.003125}
        return costs[tier]

c=RCC(0.35)
mix=([("greeting",15),("factual",20),("pricing",8),("enrollment",7),("summarize",15),
    ("code_simple",10),("analysis",8),("code_complex",5),("reasoning",7),("creative",5)])
reqs=[]
for qt,n in mix: reqs.extend([qt]*n)
random.shuffle(reqs); reqs=(reqs*3)[:200]

total=sum(c.process(q) for q in reqs)
s=c.s; ap=200*0.003125
co=(200-s["ch"])*0.003125
ro=s["fl"]*0.000170+s["f25"]*0.000255+s["pro"]*0.003125+s["ch"]*0.003125

print("Router + Cache Combo:")
print(f"  Cache: {s['ch']} | Flash: {s['fl']} | Flash25: {s['f25']} | Pro: {s['pro']}")
for name,cost in [("Always Pro",ap),("Cache only",co),("Route only",ro),("Combined",total)]:
    sv=(1-cost/ap)*100 if ap else 0
    print(f"  {name:<25} ${cost:.4f} ({sv:.0f}% saved)")
m=10000/200*30
print(f"\n  Monthly (10K/day): combined ${total*m:.2f} vs Pro ${ap*m:.2f}")


</details>


---
## Exercise 4 (Challenge): Adaptive Routing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random
from collections import defaultdict
random.seed(42)

class AR:
    def __init__(self,th=7.0):
        self.th=th
        self.rt={"greeting":"flash","factual":"flash","summarize":"flash",
            "code_simple":"flash25","analysis":"pro","reasoning":"pro"}
        self.ql=defaultdict(lambda:defaultdict(list)); self.ups=[]; self.dns=[]
    def route(self,cat): return self.rt.get(cat,"flash25")
    def record(self,cat,model,score): self.ql[cat][model].append(score)
    def adapt(self,w=10):
        ch=[]
        for cat,models in self.ql.items():
            cur=self.rt.get(cat,"flash25")
            if cur not in models: continue
            recent=models[cur][-w:]
            if len(recent)<5: continue
            avg=sum(recent)/len(recent)
            if avg<self.th:
                up={"flash":"flash25","flash25":"pro"}.get(cur)
                if up: self.rt[cat]=up; self.ups.append({"cat":cat,"from":cur,"to":up,"q":round(avg,1)}); ch.append(f"UP {cat}: {cur}->{up}")
        return ch

r=AR(7.0)
cats=["greeting","factual","summarize","code_simple","analysis","reasoning"]
bq={"greeting":{"flash":9.5},"factual":{"flash":9.0},"summarize":{"flash":8.0,"flash25":8.8},
    "code_simple":{"flash25":8.5},"analysis":{"pro":9.0},"reasoning":{"pro":9.0}}

print("Adaptive Router:")
print(f"  Initial: {dict(r.rt)}")

for i in range(100):
    cat=random.choice(cats); model=r.route(cat)
    base=bq.get(cat,{}).get(model,7.0)
    if i>=50 and cat=="summarize" and model=="flash": base=5.5
    score=max(1,min(10,base+random.gauss(0,0.5)))
    r.record(cat,model,score)
    if (i+1)%10==0:
        ch=r.adapt(10)
        if ch: print(f"  Req {i+1}: {' | '.join(ch)}")

print(f"\n  Final: {dict(r.rt)}")
print(f"  Upgrades: {len(r.ups)}")
for u in r.ups: print(f"    {u['cat']}: {u['from']}->{u['to']} (quality {u['q']}<{r.th})")


</details>
